## Applied Machine Learning Homework 3
### Shane Wang (tsw2134)

#### Note: all discussions will be in the report. 

In [1]:
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import collections

import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# from sklearn.model_selection import train_test_split

In [2]:
# Seeding for consistency
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Setting chip
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')

Using device: mps


### Data Preparation

#### a. Load Tiny Shakespeare text 

In [3]:
with open('input.txt', 'r') as f:
    raw_text = f.read()

print(raw_text[20000:20050])

the good horse is mine.

MARCIUS:
I'll buy him of 


#### b. Tokenization

Use a `subword-level tokenizer` (e.g., Byte Pair Encoding, WordPiece) to convert text into integer tokens. This captures meaningful text units (like “ing”, “the”, “tion”) rather than single characters, improving both efficiency and coherence. 

Keep the vocabulary size ≤ 500 for lightweight experiments.

In [4]:
# Using Byte Pair Encoding for encoding
VOCAB_SIZE = 500

In [5]:
# Copy input to temporary folder for training purposes
with open('/tmp/corpus.txt', 'w') as f:
    f.write(raw_text)

In [6]:
# Initialize tokenizer (BPE) and specify unknown token replacement
tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False) # probably unnecessary for Tiny Shakespeare
# tokenizer.decoder = ByteLevelDecoder() # uncomment to remove weird whitespace characters in # Testing words cell

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=['[UNK]', '[PAD]'],
    min_frequency=2
)
tokenizer.train(files=['/tmp/corpus.txt'], trainer=trainer)

# Encode corpus
encoding = tokenizer.encode(raw_text)
token_ids = encoding.ids

ACTUAL_VOCAB = tokenizer.get_vocab_size()

In [7]:
# Testing encoding success
print(f'Actual vocab size: {ACTUAL_VOCAB}')
print(f'Total tokens after encoding: {len(token_ids):,}')
print(f'Sample tokens (first 20): {token_ids[:20]}')

Actual vocab size: 500
Total tokens after encoding: 516,697
Sample tokens (first 20): [482, 231, 85, 47, 64, 90, 10, 65, 14, 43, 359, 142, 395, 119, 126, 214, 63, 82, 172, 58]


In [8]:
# Testing words
print(tokenizer.decode(token_ids[:20]))
print(tokenizer.id_to_token(482))
print(tokenizer.id_to_token(0))

First ĠC it i z en : Ċ B e fore Ġwe Ġpro ce ed Ġan y Ġf ur t
First
[UNK]


#### c. Sequence formatting
Split the tokenized text into overlapping fixed-length sequences (e.g., 50 tokens)
For each sequence:
* Input: first N tokens
* Target: same sequence shifted by one position (next-token prediction)

Example
* Input : [71, 32, 98, 101, 11, 111]
* Target : [32, 98, 101, 11, 111, 114]

In [9]:
SEQUENCE_LENGTH = 50
STRIDE = 1

In [14]:
# def make_sequences(ids, seq_len, stride):
#     inputs, targets = [], []
#     for start in range(0, len(ids) - seq_len, stride):
#         chunk = ids[start : start + seq_len + 1]
#         inputs.append(chunk[:-1])
#         targets.append(chunk[1:]) 
#     return inputs, targets

# all_inputs, all_targets = make_sequences(token_ids, SEQUENCE_LENGTH, STRIDE)

# Make sequences as input, output (target) pairs
def make_sequences(ids, seq_len, stride):
    ids_tensor = torch.tensor(ids, dtype=torch.long)
    inputs, targets = [], []
    for start in range(0, len(ids_tensor) - seq_len, stride):
        inputs.append(ids_tensor[start : start + seq_len])
        targets.append(ids_tensor[start + 1 : start + seq_len + 1])
    return torch.stack(inputs), torch.stack(targets)

all_inputs, all_targets = make_sequences(token_ids, SEQUENCE_LENGTH, STRIDE)

print(f"Input shape: {all_inputs.shape}")
print(f"Target shape: {all_targets.shape}")

Input shape: torch.Size([516647, 50])
Target shape: torch.Size([516647, 50])


#### d. Data split
Use 80% for training and 20% for validation, no test set is needed this time

In [15]:
# # The following causes data leakage:
# all_inputs, all_targets = make_sequences(token_ids, SEQ_LEN, STRIDE)

# train_inputs, val_inputs, train_targets, val_targets = train_test_split(
#     all_inputs, all_targets,
#     test_size=0.2,
#     random_state=SEED,
#     shuffle=True
# )

# print(f'Training sequences: {len(train_inputs):,}')
# print(f'Validation sequences: {len(val_inputs):,}')

split_index = int(len(token_ids) * 0.8)

train_ids = token_ids[:split_index]
val_ids = token_ids[split_index:]

train_inputs, train_targets = make_sequences(train_ids, SEQUENCE_LENGTH, STRIDE)
val_inputs, val_targets = make_sequences(val_ids, SEQUENCE_LENGTH, STRIDE)

print(f'Training sequences: {len(train_inputs):,}')
print(f'Validation sequences: {len(val_inputs):,}')
print(f"Training shape: {train_inputs.shape}")
print(f"Validation shape: {val_inputs.shape}")

Training sequences: 413,307
Validation sequences: 103,290
Training shape: torch.Size([413307, 50])
Validation shape: torch.Size([103290, 50])


#### e. Token embedding
Use `nn.Embedding` for tokens and add positional encodings

In [16]:
D_MODEL = 128  # embedding / hidden dimension

token_embedding = nn.Embedding(ACTUAL_VOCAB, D_MODEL).to(DEVICE)

# Sanity check
sample_input  = train_inputs[:4].to(DEVICE) # (4, 50)
sample_output = token_embedding(sample_input) # (4, 50, 128)

print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {sample_output.shape}")

Input shape:  torch.Size([4, 50])
Output shape: torch.Size([4, 50, 128])


### Implement a Tiny Transformer
Build a minimal Transformer-based language model including:
* Positional encoding (sinusoidal or RoPE if you want more challenges)
* Self-attention module
* Apply a causal mask to prevent the model from attending to future tokens
* Feed-forward network with residual connections and RMSnorm

Train it using `cross-entropy loss` for next-token prediction

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout) # dropping numbers to prevent overfitting

        # Build the (max_seq_len, d_model) encoding matrix once and register it as a buffer so it moves with the model (to MPS/CUDA) 
        # but is not a trainable parameter.
        positional_encoding = torch.zeros(max_seq_len, d_model) # (Time (sequence length), Dimension (d_model))
        positions = torch.arange(max_seq_len).unsqueeze(1).float() # (Time (sequence length), 1)
        # Denominator: 10000^(2i/d_model), computed in log-space for stability
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (D/2,)

        positional_encoding[:, 0::2] = torch.sin(positions * div_term) # even indices
        positional_encoding[:, 1::2] = torch.cos(positions * div_term) # odd indices

        positional_encoding = positional_encoding.unsqueeze(0)  # (1, T, D) — broadcasts over batch dimension
        self.register_buffer('pe', positional_encoding)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (Batch, Time (sequence length), Dimension (d_model))
        """
        x = x + self.positional_encoding[:, :x.size(1), :]  # slice to actual sequence length
        return self.dropout(x)

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Combines:
      1. nn.Embedding  — maps token ids → dense vectors
      2. PositionalEncoding — injects position information
    """
    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, max_seq_len, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, T)  — batch of token-id sequences
        returns (B, T, D)
        """
        # Scale embeddings by sqrt(d_model) to keep variance stable when
        # added to the O(1)-magnitude positional encodings (original paper §3.4)
        tok = self.token_emb(x) * math.sqrt(self.d_model)  # (B, T, D)
        return self.pos_enc(tok)                            # (B, T, D)


# Sanity check
embedding_layer = TokenEmbedding(vocab_size=ACTUAL_VOCAB, d_model=D_MODEL, max_seq_len=SEQUENCE_LENGTH, dropout=0.1).to(DEVICE)

# Verify shapes
sample_input = train_inputs[:4].to(DEVICE)          # (4, 50)
sample_output = embedding_layer(sample_input)        # (4, 50, 128)

print(f"Input  shape : {sample_input.shape}")        # (4, 50)
print(f"Output shape : {sample_output.shape}")       # (4, 50, 128)
print(f"Token emb weight shape: {embedding_layer.token_emb.weight.shape}")  # (500, 128)
print(f"Positional enc buffer shape: {embedding_layer.pos_enc.pe.shape}")   # (1, 50, 128)